In [1]:
# Imports nécessaires
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Charger la dataframe finale
df = pd.read_csv(r"C:\\Users\\zizou\\OneDrive\\Desktop\\stage 3ème\\day 2\\csvfiles\\dataframefinale.csv", sep=';', encoding='utf-8-sig')

# Préparation : Séparer features et target
X = df.drop(columns=['Montant'])  # Features
y = df['Montant']  # Target

# Identifier colonnes catégorielles et numériques
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()  # ex. Libellé, ISIN, JourSemaine
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Split train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Données prêtes : Train shape", X_train.shape, "Test shape", X_test.shape)

Données prêtes : Train shape (16152, 12) Test shape (4038, 12)


In [2]:
# Fonction pour calculer RMSE
def calculate_rmse(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return rmse

# Préprocesseur de base (encoding + scaling optionnel)
def get_preprocessor(with_scaling=False):
    transformers = [
        ('num', 'passthrough' if not with_scaling else StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
    return ColumnTransformer(transformers=transformers)

# Pour feature selection
def get_pipeline_with_fs(model, k=10):
    preprocessor = get_preprocessor(with_scaling=True)
    return Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(f_regression, k=k)),
        ('model', model)
    ])


In [3]:
print("=== Modèle 1: Régression Linéaire ===")

# Étape 1: Modèle de base
lr_base = Pipeline([('preprocessor', get_preprocessor()), ('model', LinearRegression())])
rmse_base = calculate_rmse(lr_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
lr_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', LinearRegression())])
rmse_scaled = calculate_rmse(lr_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
lr_fs = get_pipeline_with_fs(LinearRegression(), k=10)
rmse_fs = calculate_rmse(lr_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning (utilisation de Ridge pour régularisation)
param_grid = {'model__alpha': [0.1, 1.0, 10.0]}
lr_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', Ridge())])
grid_search_lr = GridSearchCV(lr_tuned, param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search_lr.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_lr.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning (Ridge):", rmse_tuned, "Meilleurs params:", grid_search_lr.best_params_)

=== Modèle 1: Régression Linéaire ===
Étape 1 - RMSE base: 5.9637731781707926
Étape 2 - RMSE avec normalisation: 5.895011287381392
Étape 3 - RMSE avec feature selection: 6.716441948141162
Étape 4 - RMSE après fine-tuning (Ridge): 5.822591373229153 Meilleurs params: {'model__alpha': 10.0}


In [4]:
print("=== Modèle 2: Random Forest Regressor ===")

# Étape 1: Modèle de base
rf_base = Pipeline([('preprocessor', get_preprocessor()), ('model', RandomForestRegressor(random_state=42))])
rmse_base = calculate_rmse(rf_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
rf_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', RandomForestRegressor(random_state=42))])
rmse_scaled = calculate_rmse(rf_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
rf_fs = get_pipeline_with_fs(RandomForestRegressor(random_state=42), k=10)
rmse_fs = calculate_rmse(rf_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200], 'model__max_depth': [10, 20]}
rf_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', RandomForestRegressor(random_state=42))])
grid_search_rf = GridSearchCV(rf_tuned, param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search_rf.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_rf.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_rf.best_params_)


=== Modèle 2: Random Forest Regressor ===
Étape 1 - RMSE base: 2.545110266660965
Étape 2 - RMSE avec normalisation: 2.575209252426892
Étape 3 - RMSE avec feature selection: 5.07711110196696
Étape 4 - RMSE après fine-tuning: 2.921581717732803 Meilleurs params: {'model__max_depth': 20, 'model__n_estimators': 200}


In [5]:
print("=== Modèle 3: XGBoost Regressor ===")

# Étape 1: Modèle de base
xgb_base = Pipeline([('preprocessor', get_preprocessor()), ('model', XGBRegressor(random_state=42))])
rmse_base = calculate_rmse(xgb_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
xgb_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', XGBRegressor(random_state=42))])
rmse_scaled = calculate_rmse(xgb_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
xgb_fs = get_pipeline_with_fs(XGBRegressor(random_state=42), k=10)
rmse_fs = calculate_rmse(xgb_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200], 'model__learning_rate': [0.01, 0.1]}
xgb_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', XGBRegressor(random_state=42))])
grid_search_xgb = GridSearchCV(xgb_tuned, param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search_xgb.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_xgb.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_xgb.best_params_)


=== Modèle 3: XGBoost Regressor ===
Étape 1 - RMSE base: 2.2254571950658444
Étape 2 - RMSE avec normalisation: 2.232144802657727
Étape 3 - RMSE avec feature selection: 5.48567580790965
Étape 4 - RMSE après fine-tuning: 2.3955733740322285 Meilleurs params: {'model__learning_rate': 0.1, 'model__n_estimators': 200}


In [6]:
print("=== Modèle 4: LightGBM Regressor ===")

# Étape 1: Modèle de base
lgbm_base = Pipeline([('preprocessor', get_preprocessor()), ('model', LGBMRegressor(random_state=42, verbose=-1))])
rmse_base = calculate_rmse(lgbm_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation
lgbm_scaled = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', LGBMRegressor(random_state=42, verbose=-1))])
rmse_scaled = calculate_rmse(lgbm_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
lgbm_fs = get_pipeline_with_fs(LGBMRegressor(random_state=42, verbose=-1), k=10)
rmse_fs = calculate_rmse(lgbm_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__n_estimators': [100, 200], 'model__learning_rate': [0.01, 0.1]}
lgbm_tuned = Pipeline([('preprocessor', get_preprocessor(with_scaling=True)), ('model', LGBMRegressor(random_state=42, verbose=-1))])
grid_search_lgbm = GridSearchCV(lgbm_tuned, param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search_lgbm.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_lgbm.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_lgbm.best_params_)


=== Modèle 4: LightGBM Regressor ===
Étape 1 - RMSE base: 2.4989064766251947
Étape 2 - RMSE avec normalisation: 2.5110823842814907
Étape 3 - RMSE avec feature selection: 5.283508897639026
Étape 4 - RMSE après fine-tuning: 2.367907388397989 Meilleurs params: {'model__learning_rate': 0.1, 'model__n_estimators': 200}


In [8]:
print("=== Modèle 5: CatBoost Regressor ===")

# Étape 1: Modèle de base
# Préprocesseur pour CatBoost : passe les colonnes numériques et catégorielles telles quelles
preprocessor_cat = ColumnTransformer([
    ('num', 'passthrough', numerical_cols),
    ('cat', 'passthrough', categorical_cols)  # Pas d'encodage pour les cat. features
])
cat_base = Pipeline([
    ('preprocessor', preprocessor_cat),
    ('model', CatBoostRegressor(random_state=42, verbose=0, cat_features=categorical_cols))
])
rmse_base = calculate_rmse(cat_base, X_train, y_train, X_test, y_test)
print("Étape 1 - RMSE base:", rmse_base)

# Étape 2: Avec normalisation (uniquement sur colonnes numériques)
preprocessor_cat_scaled = ColumnTransformer([
    ('num', StandardScaler(), numerical_cols),
    ('cat', 'passthrough', categorical_cols)  # Pas d'encodage pour les cat. features
])
cat_scaled = Pipeline([
    ('preprocessor', preprocessor_cat_scaled),
    ('model', CatBoostRegressor(random_state=42, verbose=0, cat_features=categorical_cols))
])
rmse_scaled = calculate_rmse(cat_scaled, X_train, y_train, X_test, y_test)
print("Étape 2 - RMSE avec normalisation:", rmse_scaled)

# Étape 3: Avec feature selection
# Pour feature selection, on encode les cat. features temporairement pour SelectKBest
cat_fs = Pipeline([
    ('preprocessor', get_preprocessor(with_scaling=True)),  # Utilise OneHotEncoder ici
    ('feature_selection', SelectKBest(f_regression, k=10)),
    ('model', CatBoostRegressor(random_state=42, verbose=0))  # Pas de cat_features ici
])
rmse_fs = calculate_rmse(cat_fs, X_train, y_train, X_test, y_test)
print("Étape 3 - RMSE avec feature selection:", rmse_fs)

# Étape 4: Fine-tuning
param_grid = {'model__iterations': [100, 200], 'model__learning_rate': [0.01, 0.1]}
cat_tuned = Pipeline([
    ('preprocessor', preprocessor_cat_scaled),
    ('model', CatBoostRegressor(random_state=42, verbose=0, cat_features=categorical_cols))
])
grid_search_cat = GridSearchCV(cat_tuned, param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search_cat.fit(X_train, y_train)
rmse_tuned = np.sqrt(mean_squared_error(y_test, grid_search_cat.predict(X_test)))
print("Étape 4 - RMSE après fine-tuning:", rmse_tuned, "Meilleurs params:", grid_search_cat.best_params_)

=== Modèle 5: CatBoost Regressor ===


CatBoostError: features parameter contains string value 'dataloadingdate' but feature names for a dataset are not specified